**Análisis de Datos e Inteligencia Artificial en el Sector Eléctrico**
===================================

<table>
<tr><td>
<img src='https://enae.cecacier.org/wp-content/uploads/2024/09/Logo-CECACIER-2048x706.png' width='520'></td>
<td>
<img src='https://www.univalle.edu.co/images/Univalle/70_Rojo.jpg' width='140'></td>
</tr>
</table>

**MÓDULO 3.7. Power BI y Claude - Automatización de Informes mediante MCP**

<table>
<tr><th>Instructores:</th><th><i>Wilfredo Alfonso-Morales, PhD</i></th></tr>
<tr><th></th><th><i>Eduardo F. Caicedo-Bravo, PhD</i></th></tr>
</table>

---
# 1. Introducción

---

> **Alcance del manual técnico:**
> Esta guía proporciona los lineamientos de arquitectura y ejecución técnica para integrar asistentes de inteligencia artificial (Claude Desktop y Claude Code) con **Power BI Desktop**, utilizando el protocolo estandarizado **Model Context Protocol (MCP)** y el ecosistema **Microsoft Fabric Skills**.
>
> El flujo de trabajo aborda el ciclo de vida de un producto analítico a partir del caso de estudio **EcoBici Lima** (Red de bicicletas públicas 2023-2025):
> 1. Auditoría de calidad, perfilado y consistencia de datos fuente.
> 2. Construcción programática del modelo semántico en estrella (parámetro `RutaDatos`, tablas `fct` y `dim`, relaciones activas/inactivas y dimensión temporal).
> 3. Desarrollo y categorización jerárquica de medidas DAX (Base, Avanzadas con contexto relacional y Time Intelligence).
> 4. Despliegue de un sistema de diseño mediante tokens JSON (`Tema_EcoBici.json`) y segmentadores sincronizados.
> 5. Generación y validación de tres páginas analíticas: **Resumen Operacional**, **Análisis de Uso** y **Tendencias**.

---

### A. _Arquitectura de Interconexión_

```text
+-------------------------+          +--------------------------------------------------------+
|      Claude Desktop     |          |             Servidores Locales de Protocolo            |
|   (Chat + Claude Code)  | <------> | 1. MCP de Modelado (powerbi-modeling-mcp.exe local)  |
+-------------------------+          | 2. Servidor MCP Filesystem (@modelcontextprotocol/fs)  |
                                     | 3. CLI Bridge (@microsoft/powerbi-desktop-bridge-cli)  |
                                     +---------------------------+----------------------------+
                                                                 |
                                                                 v
                                             +---------------------------------------+
                                             |           Power BI Desktop            |
                                             |   (API Local Segura + Archivo .pbip)  |
                                             +---------------------------------------+
```

---

### B. _Recursos del Caso de Estudio (EcoBici Lima)_
* **Repositorio de datos fuente:** Directorio `Datos_Fuente` con el conjunto transaccional `Viajes.xlsx`, maestros en formato CSV de estaciones, `Mapa_Zonas` y el archivo de especificación temática `Tema_EcoBici.json`.
* **Enlace de descarga de recursos:** [Acceso al repositorio de archivos en Google Drive](https://drive.google.com/drive/folders/1YX51a_8xc5xK977NZc3iW8tV00mX3buD?usp=sharing)
* **Entorno operativo:** Estaciones de trabajo Windows 10/11 de 64 bits con Power BI Desktop actualizado.

# 2. Configuración del Entorno de Trabajo

## 2.1. Requisitos de Software y Componentes Base

&nbsp; Para habilitar la infraestructura de integración local, se requiere el despliegue del siguiente stack tecnológico:

| Componente | Versión Recomendada | Enlace Oficial de Descarga | Función en la Arquitectura |
| :--- | :--- | :--- | :--- |
| **Node.js** | **LTS (Long Term Support)** | [nodejs.org/es/download](https://nodejs.org/es/download) | Entorno de ejecución requerido por `npx`, con el que se lanza el servidor MCP de archivos, y por `npm` para las herramientas CLI. |
| **Claude Desktop** | Última versión disponible | [claude.com/es/download](https://claude.com/es/download) | Cliente principal para agentes de IA y ejecución de Claude Code. |
| **Power BI Desktop** | Última versión de 64 bits | [microsoft.com/power-bi/desktop](https://www.microsoft.com/es-co/power-platform/products/power-bi/desktop) | Motor tabular local (Analysis Services) y capa de renderizado visual. |
| **Visual Studio Code** | Distribución estándar | [code.visualstudio.com/download](https://code.visualstudio.com/download) | Solo alternativa B: aloja la extensión que instala el binario `powerbi-modeling-mcp.exe`. |

---

### A. _Consideraciones Críticas de Instalación_
1. **Configuración de variables de entorno:** Al instalar Node.js, es imperativo habilitar la opción **"Add to PATH"**. Sin esta variable registrada a nivel de sistema, los clientes de IA no podrán instanciar procesos secundarios de `npx`.
2. Realizar un lanzamiento inicial de **Claude Desktop** y **VS Code** para inicializar los perfiles de usuario y directorios de configuración locales.

---

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/NodejsDownload.png' width='700px'><br>
<caption><b>Figura 1<a name="fig:01"></a>.</b> Portal oficial de descarga de Node.js destacando el instalador precompilado de Windows (.msi) en su versión LTS. </caption>
</center>

---

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/Claude.png' width='700px'><br>
<caption><b>Figura 2<a name="fig:02"></a>.</b> Captura de pantalla del cliente Claude Desktop en Windows tras inicializar la sesión. </caption>
</center>

---

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/VsCode1.png' width='700px'><br>
<caption><b>Figura 3<a name="fig:03"></a>.</b> Captura de pantalla de la interfaz de Visual Studio Code en Windows. </caption>
</center>

## 2.2. Validación del Entorno Node.js y Gestores de Paquetes

&nbsp; Antes de proceder con la configuración de protocolos, se debe auditar la disponibilidad de los ejecutables de Node.js en la consola del sistema.

### A. _Comprobación en Terminal_
1. Ejecutar una consola de comandos (**PowerShell** o **CMD**).
2. Consultar las versiones registradas:

```bash
node -v
npm -v
npx -v
```

3. El sistema debe responder con versiones compatibles (Node.js >= v20.x y npm >= 10.x).

> **Criterio de resolución técnica:**  
> Si la terminal emite un error de comando no reconocido, reiniciar la terminal. De persistir la incidencia en estaciones de trabajo con políticas de permisos restrictivas, ejecutar el instalador en modo **Repair** validando permisos de modificación sobre variables de entorno de usuario y sistema.

---

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/cmd.png' width='700px'><br>
<caption><b>Figura 4<a name="fig:04"></a>.</b> Captura de pantalla de PowerShell o CMD ejecutando las sentencias de validación con las versiones de Node.js, npm y npx desplegadas en pantalla. </caption>
</center>

## 2.3. Instalación Global de Herramientas CLI de Microsoft

&nbsp; Para posibilitar la inspección en tiempo real, recargas en caliente (*hot-reload*) y validación de sintaxis de reportes PBIR, se instalan los toolkits oficiales desarrollados por Microsoft.

### A. _Despliegue de Paquetes Globales_
Ejecutar la siguiente instrucción en una consola de comandos con privilegios suficientes:

```bash
npm install -g @microsoft/powerbi-desktop-bridge-cli @microsoft/powerbi-report-authoring-cli
```

---

### B. _Especificación Técnica de los Componentes_
* **`@microsoft/powerbi-desktop-bridge-cli` (`powerbi-desktop`):** Expone un puente de comunicación IPC/HTTP seguro con la instancia en memoria de Power BI Desktop, habilitando comandos de recarga de esquema y captura automatizada de estados visuales.
* **`@microsoft/powerbi-report-authoring-cli` (`powerbi-report-author`):** Valida esquemas JSON contra el estándar PBIR, verifica identificadores únicos y permite la compilación y prueba de especificaciones visuales programáticas.

---

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/Captura%20de%20pantalla%202026-10-26%20150537.png?raw=true' width='700px'><br>
<caption><b>Figura 5<a name="fig:05"></a>.</b> Captura de pantalla de la terminal mostrando la salida de npm install -g con los paquetes de Microsoft registrados satisfactoriamente en el PATH global. </caption>
</center>

## 2.4. Instalación de la Extensión Power BI Modeling MCP en VS Code

&nbsp; Este paso es necesario únicamente si se elige la **alternativa B** de configuración, que ejecuta el servidor de modelado a partir del binario local `powerbi-modeling-mcp.exe`, instalado por la extensión oficial de Microsoft para Visual Studio Code. Con la **alternativa A** (`npx`) puede omitirse. VS Code se utiliza únicamente para esta instalación y no participa en la interacción con la IA.

### A. _Procedimiento de Despliegue_
1. Iniciar **Visual Studio Code**.
2. Acceder al panel de extensiones (`Ctrl + Shift + X`).
3. Buscar la extensión oficial: `Power BI Modeling MCP`.
4. Validar que el publicador oficial sea **Microsoft** y proceder con la instalación.
5. Concluida la instalación, se puede cerrar VS Code. El ejecutable queda disponible en `%USERPROFILE%\.vscode\extensions` y se encuentra ubicado en
analysis-services.powerbi-modeling-mcp-1.0.0-win32-x64\server.

---

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/InstalacionMCPVS.png?raw=true' width='700px'><br>
<caption><b>Figura 6<a name="fig:06"></a>.</b> Captura de pantalla del Marketplace de extensiones en VS Code con el componente "Power BI Modeling MCP" de Microsoft en estado instalado. </caption>
</center>

---

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/DirMCP.png?raw=true' width='700px'><br>
<caption><b>Figura 7<a name="fig:07"></a>.</b> En la carpeta se puede observar la dirección del MCP que se usara. </caption>
</center>

## 2.5. Estructura del Proyecto y Aislamiento de Seguridad (Sandboxing)

&nbsp; El servidor de archivos confina el alcance de lectura y escritura de Claude exclusivamente al directorio designado, mitigando riesgos de acceso no autorizado en la estación de trabajo. Esta carpeta reúne los datos de origen y el proyecto de Power BI.

### A. _Topología de Directorios_
1. Crear un directorio raíz para el proyecto analítico (por ejemplo en el Escritorio), denominado `PowerBI`.
2. Disponer la siguiente estructura interna:
   * **`Datos_Fuente/`**: Directorio donde residen los activos crudos (`Viajes.xlsx`, CSVs maestros de estaciones, `Mapa_Zonas` y `Tema_EcoBici.json`).
   * **`informe/`**: Directorio reservado para el repositorio de código del informe en formato `.pbip`.
3. Obtener la ruta absoluta canónica desde el Explorador de Windows.

&nbsp; Estructura de referencia:
```text
C:\Users\TuUsuario\Desktop\PowerBI\
│
├── Datos_Fuente\                  <- Activos de datos crudos y especificación de diseño
└── informe\                       <- Directorio del proyecto modular EcoBici_Lima.pbip
```

> **Directriz de formateo para Windows:** Al declarar rutas dentro de configuraciones JSON, los separadores deben escribirse con doble barra invertida (`\\`). Esto aplica a la ruta del ejecutable del servidor MCP y a la ruta de la carpeta de trabajo (`C:\\Users\\TuUsuario\\Desktop\\PowerBI`).

---

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/FilePowerBI.png?raw=true' width='700px'><br>
<caption><b>Figura 8<a name="fig:08"></a>.</b> Captura de pantalla del Explorador de archivos mostrando el directorio raíz del proyecto con las subcarpetas Datos_Fuente e informe debidamente estructuradas. </caption>
</center>

## 2.6. Configuración de Servidores MCP en Claude Desktop

&nbsp; Se realiza el enlace entre el cliente conversacional y los dos servidores locales: uno gestiona el modelo semántico de Power BI y el otro da acceso controlado a la carpeta de trabajo.

---

### A. _Acceso al Archivo de Manifiesto_

1. En Claude Desktop, acceder a la configuración de usuario (esquina inferior izquierda) y seleccionar **Ajustes** (Settings). Ver [Figura 9](#fig:09).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/ClaudeConfig.png' width='700px'><br>
<caption><b>Figura 9<a name="fig:09"></a>.</b> Acceso a Ajustes de Claude Desktop. </caption>
</center>

2. Navegar a la sección **Desarrollador** (Developer) y seleccionar **Editar configuración** (Edit Config). Ver [Figura 10](#fig:10).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-powerbi-mcp/06-desarrollador-editar-config.png' width='700px'><br>
<caption><b>Figura 10<a name="fig:10"></a>.</b> Panel Desarrollador en Claude Desktop. </caption>
</center>

3. El sistema abrirá la ubicación de persistencia con el archivo `claude_desktop_config.json`. Ver [Figura 11](#fig:11).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/01-explorador-carpeta-config-claude.png' width='700px'><br>
<caption><b>Figura 11<a name="fig:11"></a>.</b> Directorio del Archivo de Configuración. </caption>
</center>

4. Abrir `claude_desktop_config.json` con un editor de texto plano o editor de código. Ver [Figura 12](#fig:12).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-powerbi-mcp/07-editar-bloc-de-notas.png' width='700px'><br>
<caption><b>Figura 12<a name="fig:12"></a>.</b> Edición en Editor de Texto. </caption>
</center>

---

### B. _Configuración MCP_

&nbsp; El archivo `claude_desktop_config.json` suele contener otras claves propias de la aplicación (por ejemplo `preferences` o `coworkUserFilesPath`). **No se debe reemplazar el contenido completo del archivo**: únicamente se agrega la clave `mcpServers` al mismo nivel que las demás, separada por coma. Si el archivo está vacío o solo contiene `{}`, el bloque puede usarse tal cual.

&nbsp; Se presentan dos alternativas completas y autocontenidas. Debe elegirse **una sola**; no se combinan ni una corrige a la otra.

| | Alternativa A: `npx` | Alternativa B: ejecutable local |
| :--- | :--- | :--- |
| Servidor de Power BI | `npx -y @microsoft/powerbi-modeling-mcp@latest` | `powerbi-modeling-mcp.exe` de la extensión de VS Code |
| Requiere VS Code | No | Sí, con la extensión instalada |
| Requiere Node.js | Sí | Sí, por el servidor `archivos` |
| Ruta local específica | No | Sí, varía por usuario y por versión de la extensión |
| Uso recomendado | Distribución general | Equipos donde `npx` no logra arrancar el servidor de Power BI |

&nbsp; En ambas alternativas, el servidor `archivos` se ejecuta con `npx` y da acceso a la carpeta de trabajo desde el chat de Claude Desktop, sin necesidad de Claude Code (utilizando el entorno Node.js validado previamente).

**Alternativa A: ejecución con `npx`**

```json
{
  "mcpServers": {
    "powerbi-modeling-mcp": {
      "command": "npx",
      "args": ["-y", "@microsoft/powerbi-modeling-mcp@latest", "--start"],
      "env": {
        "PBI_MODELING_MCP_ACCEPT_EULA": "true"
      }
    },
    "archivos": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "C:\\Users\\TU_USUARIO\\Desktop\\PowerBI"
      ]
    }
  }
}
```

**Alternativa B: ejecutable local de la extensión de VS Code**

```json
{
  "mcpServers": {
    "powerbi-modeling-mcp": {
      "command": "C:\\Users\\TU_USUARIO\\.vscode\\extensions\\analysis-services.powerbi-modeling-mcp-1.0.0-win32-x64\\server\\powerbi-modeling-mcp.exe",
      "args": [
        "--start"
      ],
      "env": {
        "PBI_MODELING_MCP_ACCEPT_EULA": "true"
      }
    },
    "archivos": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "C:\\Users\\TU_USUARIO\\Desktop\\PowerBI"
      ]
    }
  }
}
```

&nbsp; Valores que deben ajustarse a cada equipo:
* **Ambas alternativas:** último argumento de `archivos`, es decir la ruta de la carpeta de trabajo definida en la sección de estructura del proyecto.
* **Solo la alternativa B:** `command` de `powerbi-modeling-mcp`, con la ruta completa del ejecutable. El nombre de la carpeta incluye la versión de la extensión (en el ejemplo `1.0.0`), que puede diferir.

&nbsp; Recomendaciones al editar el archivo:
* Pegar el bloque inmediatamente después de abrir el archivo con **Editar configuración**. Si se pega una copia antigua del archivo, se pueden perder preferencias que la aplicación haya modificado entre tanto.
* No compartir el archivo completo: la sección `preferences` contiene identificadores de cuenta y organización. Para compartir la configuración, entregar únicamente el bloque `mcpServers`.

---

### C. _Ubicación del Ejecutable `powerbi-modeling-mcp.exe` (solo alternativa B)_

### D. _Método 1. Mediante PowerShell, el siguiente comando devuelve la ruta completa_

```powershell
Get-ChildItem "$env:USERPROFILE\.vscode\extensions" -Recurse -Filter powerbi-modeling-mcp.exe | Select-Object -ExpandProperty FullName
```

### E. _Método 2. Mediante el Explorador de Windows_
1. En la barra de direcciones, escribir `%USERPROFILE%\.vscode\extensions`.
2. Abrir la carpeta que comienza con `analysis-services.powerbi-modeling-mcp-` y entrar a `server`.
3. Clic derecho sobre `powerbi-modeling-mcp.exe` (o `Shift` + clic derecho) y seleccionar **Copiar como ruta de acceso**. Ver [Figura 13](#fig:13).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-powerbi-mcp/04-copiar-ruta-exe.png' width='700px'><br>
<caption><b>Figura 13<a name="fig:13"></a>.</b> Opción "Copiar como ruta de acceso" sobre el archivo powerbi-modeling-mcp.exe dentro de la carpeta server de la extensión. </caption>
</center>

&nbsp; La ruta copiada utiliza una sola barra invertida. Dentro del JSON, cada `\` debe escribirse como `\`.

> **Aspectos técnicos relevantes:**
> * `PBI_MODELING_MCP_ACCEPT_EULA`: en `"true"` suprime la confirmación interactiva de los términos de licencia durante el arranque del proceso.
> * La sintaxis JSON debe conservar llaves, comas y comillas; una coma faltante entre `mcpServers` y las demás claves invalida todo el archivo.

---

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/block.png?raw=true' width='700px'><br>
<caption><b>Figura 14<a name="fig:14"></a>.</b> Vista del archivo claude_desktop_config.json con el bloque mcpServers declarado y las demás claves del archivo conservadas. </caption>
</center>

## 2.7. Reinicio del Servicio y Validación de Estado Operativo

&nbsp; El cliente Claude Desktop mantiene procesos en segundo plano al cerrar la ventana principal; por ende, se debe forzar una finalización completa de la instancia.

### A. _Secuencia de Reinicio_
1. Dirigirse al área de notificación de la barra de tareas de Windows (system tray).
2. Efectuar clic derecho sobre el icono de Claude y pulsar **Salir** (Quit). Ver [Figura 15](#fig:15).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-powerbi-mcp/08-salir-bandeja.png' width='700px'><br>
<caption><b>Figura 15<a name="fig:15"></a>.</b> Menú contextual del icono de Claude en el área de notificación ejecutando la acción Salir. </caption>
</center>

3. Ejecutar nuevamente **Claude Desktop**.
4. Dirigirse a **Ajustes -> Desarrollador**. Ambos servidores deben presentar el indicador de estado **Running**:
   * **`powerbi-modeling-mcp`:** Running
   * **`archivos`:** Running

---

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-powerbi-mcp/09-mcp-running.png' width='700px'><br>
<caption><b>Figura 16<a name="fig:16"></a>.</b> Panel de estado en Ajustes de Desarrollador confirmando que powerbi-modeling-mcp y archivos se encuentran en ejecución activa. </caption>
</center>

## 2.8. Aprovisionamiento de Microsoft Fabric Skills

&nbsp; Las Fabric Skills dotan al modelo de lenguaje del entendimiento semántico y las directrices técnicas para la generación de estructuras PBIR, validación de modelos tabulares y mejores prácticas de optimización DAX.

### A. _Procedimiento de Instalación_
1. En Claude Desktop, acceder a la pestaña **Code** o al panel de Plugins en Ajustes.
2. Seleccionar **Añadir marketplace -> Añadir desde un repositorio**.
3. Suministrar el endpoint del repositorio oficial de Microsoft:
   ```text
   https://github.com/microsoft/skills-for-fabric
   ```
4. Completar la sincronización y activar los módulos:
   * **Power BI Authoring**
   * **Fabric Skills**

### B. _Configuración e instalación de plugins en Claude Code_

1. Vamos a la sección de plugins en la configuración de Claude Code. Ver [Figura 17](#fig:17).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/Plugins.png' width='700px'><br>
<caption><b>Figura 17<a name="fig:17"></a>.</b> Sección de plugins en Claude Code. </caption>
</center>

2. Desplegamos el menú de agregar y hacemos clic en **Agregar marketplace**. Ver [Figura 18](#fig:18).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PluginsMarketplace.png?raw=true' width='700px'><br>
<caption><b>Figura 18<a name="fig:18"></a>.</b> Desplegar menú Agregar marketplace. </caption>
</center>

3. Hacemos clic en la sección que dice **Agregar desde un repositorio**. Ver [Figura 19](#fig:19).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PluginsMarketplace2.png?raw=true' width='700px'><br>
<caption><b>Figura 19<a name="fig:19"></a>.</b> Seleccionar agregar desde un repositorio. </caption>
</center>

4. Colocamos la URL `https://github.com/microsoft/skills-for-fabric` y hacemos clic en **Sincronizar**. Ver [Figura 20](#fig:20).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PluginsMarketplace3.png?raw=true' width='700px'><br>
<caption><b>Figura 20<a name="fig:20"></a>.</b> Ingresar URL del repositorio y sincronizar. </caption>
</center>

5. Hacemos clic en agregar para incluir los plugins de **Power BI Authoring** y **Fabric Skills**. Ver [Figura 21](#fig:21).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PluginsMarketplace4.png?raw=true' width='700px'><br>
<caption><b>Figura 21<a name="fig:21"></a>.</b> Agregar plugins de Power BI Authoring y Fabric Skills. </caption>
</center>

## 2.9. Habilitación de API Local en Power BI y Formato de Proyecto (.pbip)

&nbsp; Para que los agentes de software puedan comunicarse con el proceso local e inspeccionar metadatos en vivo, se deben activar los canales de API local segura y persistir el desarrollo en formato de ingeniería desacoplado.

**Nota: Las nuevas versiones de Power BI incluyen esta configuración de forma predeterminada.**

### A. _Configuración de políticas de acceso en Power BI para versiones anteriores_
1. Abrir **Power BI Desktop** con un archivo en blanco.
2. Ingresar a: **Archivo -> Opciones y configuración -> Opciones**.
3. En la sección **Global**, seleccionar **Características en versión preliminar**.
4. Habilitar obligatoriamente:
   * **Habilitar el acceso de herramientas externas a Power BI Desktop mediante API locales seguras**.
   * **Objetos visuales de mapa y mapa coropléjico** (requerido para los componentes geoespaciales de Azure Maps).
5. Confirmar con **Aceptar**.

---
1. Abrir **Power BI Desktop** con un archivo en blanco.
- **Hacemos clic en la pestaña "Archivo" en la esquina superior izquierda de Power BI Desktop.**

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig.png?raw=true' width='700px'><br>
<caption><b>Figura 22<a name="fig:22"></a>.</b> Abrir menú Archivo en Power BI Desktop. </caption>
</center>

2. Ingresar a opciones y configuración.
- **Seleccionamos "Opciones y configuración" y luego ingresamos a "Opciones".**

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig2.png?raw=true' width='700px'><br>
<caption><b>Figura 23<a name="fig:23"></a>.</b> Seleccionar Opciones y configuración. </caption>
</center>

3. Seleccionar las características en versión preliminar.
- **En el panel izquierdo, bajo la sección Global, seleccionamos "Características en versión preliminar".**

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig3_1.png?raw=true' width='700px'><br>
<caption><b>Figura 24<a name="fig:24"></a>.</b> Seleccionar Características en versión preliminar. </caption>
</center>

4. Habilitar obligatoriamente las herramientas necesarias.
- **Habilitamos la opción de guardar proyectos Power BI (.pbip) y el acceso mediante API locales seguras.**

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig4.png?raw=true' width='700px'><br>
<caption><b>Figura 25<a name="fig:25"></a>.</b> Habilitar características requeridas para API local y proyectos PBIP. </caption>
</center>

---

**Para las versiones actuales de Power BI, la interfaz se mostrará tal como se indica a continuación. Dicha característica ya se encuentra habilitada por defecto, por lo que los pasos previos no son requeridos.**

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig5.png?raw=true' width='700px'><br>
<caption><b>Figura 26<a name="fig:26"></a>.</b> Configuración habilitada por defecto en versiones actuales de Power BI. </caption>
</center>

### B. _Guardado en Formato Proyecto de Power BI (.pbip)_
1. Seleccionar **Archivo -> Guardar como**.
2. Desplegar el selector de tipo y elegir: **Proyecto de Power BI (*.pbip)** *(no utilizar el formato binario tradicional .pbix)*.
3. Guardar en el directorio `informe` con el identificador `EcoBici_Lima.pbip`:  
   `C:\Users\TuUsuario\Desktop\PowerBI\informe\EcoBici_Lima.pbip`

---

1. Seleccionar **Archivo -> Guardar como**. Ver [Figura 27](#fig:27).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBIConfig.png?raw=true' width='700px'><br>
<caption><b>Figura 27<a name="fig:27"></a>.</b> Abrir menú Archivo y seleccionar Guardar como en Power BI Desktop. </caption>
</center>

2. Elegir la opción de examinar en este dispositivo para seleccionar la ubicación de destino. Ver [Figura 28](#fig:28).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBISaves.png?raw=true' width='700px'><br>
<caption><b>Figura 28<a name="fig:28"></a>.</b> Seleccionar opción de examinar este dispositivo para guardar copia. </caption>
</center>

3. Navegar dentro del directorio de trabajo y seleccionar la carpeta destinada para el informe. Ver [Figura 29](#fig:29).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBISaves2.png?raw=true' width='700px'><br>
<caption><b>Figura 29<a name="fig:29"></a>.</b> Navegar y seleccionar la carpeta de destino en el directorio de trabajo. </caption>
</center>

4. Ingresar al directorio `informe` para proceder con el almacenamiento estructurado. Ver [Figura 30](#fig:30).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBISaves3.png?raw=true' width='700px'><br>
<caption><b>Figura 30<a name="fig:30"></a>.</b> Acceder al directorio del informe. </caption>
</center>

5. Asignar el nombre `EcoBici_Lima` y desplegar el selector de tipo para elegir obligatoriamente **Archivos de proyecto Power BI (*.pbip)** *(evitando el formato binario tradicional .pbix)*, guardando en la ruta:
   `C:\Users\TuUsuario\Desktop\PowerBI\informe\EcoBici_Lima.pbip`

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-powerbi-mcp/PowerBISaves4.png?raw=true' width='700px'><br>
<caption><b>Figura 31<a name="fig:31"></a>.</b> Seleccionar formato de proyecto pbip y asignar nombre EcoBici_Lima. </caption>
</center>

> **Concepto clave:**  
> El formato `.pbip` desglosa el informe en artefactos de texto plano: el modelo semántico se serializa en TMDL (Tabular Model Definition Language) y los informes en PBIR (Power BI Report format en JSON). Esta arquitectura permite control de versiones mediante Git y autoría desatendida mediante agentes de IA.

# 3. Taller - Construcción del Informe EcoBici Lima

## 3.1. Inicialización del Enlace MCP con el Informe Activo

&nbsp; Con el proyecto `EcoBici_Lima.pbip` guardado y abierto en Power BI Desktop (sección 2.9) y los servidores `powerbi-modeling-mcp` y `archivos` en estado **Running** (sección 2.7), se establece la sesión de control con el agente de IA.

&nbsp; Todo el desarrollo del informe se realiza en **un único chat** de Claude Desktop: los prompts de esta guía se pegan uno tras otro, en orden, esperando a que Claude termine cada respuesta. Las reglas que se dan al inicio se mantienen para todo el chat, por eso los últimos prompts son más cortos.

1. En Claude Desktop, iniciar un **nuevo chat**.

2. En el icono de herramientas (**+**) junto al cuadro de texto, dejar activos solo los conectores `powerbi-modeling-mcp` y `archivos` (desactivar búsqueda web y otros conectores).

3. Pegar el prompt de conexión:

**PROMPT 1 (Establecimiento de Conexión y Reglas de Trabajo)**
```text
Vamos a construir en este chat un informe de Power BI completo usando solo las herramientas MCP (powerbi-modeling-mcp y archivos). Reglas para todo el chat: trabaja directo, no uses la operación Help, agrupa en una sola llamada todo lo que la herramienta permita (listas), no muestres código ni JSON y responde siempre en pocas líneas.
Conéctate por MCP al informe de Power BI Desktop que tengo abierto (EcoBici_Lima) y responde solo con el nombre del modelo y el puerto.
```

4. Cuando Claude solicite permiso para usar la herramienta MCP, pulsar **Permitir siempre**.

5. Verificar que Claude responde con el nombre del modelo y el puerto local de Analysis Services. Ver [Figura 32](#fig:32).

<center>
<img src='https://github.com/eleventhcurve91/Img-repo/blob/main/claude-free/new.png?raw=true' width='700px'><br>
<caption><b>Figura 32<a name="fig:32"></a>.</b> Respuesta de Claude al PROMPT 1 confirmando la conexión con el modelo de Power BI. </caption>
</center>

> **Concepto clave:**
> Power BI Desktop levanta en segundo plano una instancia local de Analysis Services en un puerto dinámico; el servidor MCP la localiza y se conecta a la base de datos tabular activa. Trabajar en un solo chat conserva el contexto (reglas, nombres del modelo, formato PBIR) y agrupar operaciones reduce el número de llamadas a las herramientas.

## 3.2. Auditoría Técnica y Perfilado de Datos de Origen

&nbsp; Antes de articular transformaciones en el motor de Power Query, se ejecuta una auditoría sobre el directorio `Datos_Fuente` para identificar tipologías de datos, anomalías y requerimientos de estandarización.

1. En el mismo chat, pegar el prompt:

**PROMPT 2 (Perfilado y Diagnóstico de Calidad)**
```text
Con la herramienta de archivos lee ÚNICAMENTE Datos_Fuente\Estaciones_Centro.csv, Estaciones_Sur.csv y Usuarios.csv (separador ";"). Lo que ya sé de los Excel (no los abras):
- Viajes.xlsx: hojas 2023, 2024, 2025. Columnas IdViaje, InicioViaje, FinViaje (texto "dd/mm/aaaa hh:mm"), DuracionMin, IdEstacionOrigen, IdEstacionDestino, IdBicicleta, IdUsuario, DistanciaKm, Notas.
- Bicicletas.xlsx (hoja Bicicletas): IdBicicleta, CodigoBici, Tipo (Eléctrica/Mecánica), Modelo, FechaAdquisicion, RegistroInterno.
- Mapa_Zonas.xlsx (hoja Zonas): Distrito, Zona.
Responde con un resumen de calidad de datos en máximo 10 viñetas (problema → transformación propuesta) y el esquema en estrella propuesto en 3 líneas.
```

2. Revisar que el diagnóstico identifique:
   * La estructura multi-hoja de `Viajes.xlsx` (una hoja por año) y las fechas en texto `dd/mm/aaaa hh:mm`.
   * El `Distrito` de las estaciones con espacios y mayúsculas distintas (`MIRAFLORES`, `  Cercado de Lima `), que debe cruzarse con `Mapa_Zonas`.
   * El `Genero` con códigos (`M`, `F`) y vacíos (90 de 500 usuarios), que requiere el valor *No especificado*.
   * Las columnas sobrantes `Notas` y `RegistroInterno`.

> **Concepto clave:**
> Los archivos `.xlsx` son binarios y el servidor `archivos` solo lee texto; por eso el prompt entrega su estructura y Claude perfila directamente los CSV. Detectar los problemas de calidad antes de modelar evita reprocesar tablas en Power Query.

## 3.3. Construcción Programática del Modelo Dimensional (Esquema en Estrella)

&nbsp; Se ejecuta la instrucción de modelado semántico que genera el parámetro de entorno `RutaDatos`, procesa las tablas de hechos y dimensiones, y articula la topología de relaciones tabulares.

1. En el mismo chat, pegar el prompt:

**PROMPT 3 (Generación de Modelo, Parámetro y Relaciones)**
```text
Genera el modelo de datos en el Power BI abierto:
1) Parámetro de Power Query "RutaDatos" (texto, obligatorio) = ruta absoluta de la carpeta Datos_Fuente terminada en "\" (obtenla con list_allowed_directories de la herramienta de archivos). Todas las tablas leen desde RutaDatos.
2) fctViajes (M): Excel.Workbook(File.Contents(RutaDatos & "Viajes.xlsx"), true), filtra Kind = "Sheet", Table.Combine de las hojas; quita Notas (MissingField.Ignore); InicioViaje y FinViaje: si son texto DateTime.FromText(_, [Format="dd/MM/yyyy HH:mm"]), tipo datetime; enteros IdViaje, DuracionMin, IdEstacionOrigen, IdEstacionDestino, IdBicicleta, IdUsuario; DistanciaKm decimal con cultura "en-US"; añade FechaInicio = DateTime.Date([InicioViaje]) (date) y HoraInicio = Time.Hour([InicioViaje]) (entero).
3) dimEstaciones (M): Csv.Document de Estaciones_Centro.csv y Estaciones_Sur.csv (Delimiter=";", Encoding=65001), PromoteHeaders, Table.Combine. IdEstacion y Capacidad enteros, NombreEstacion con Text.Trim, Latitud y Longitud decimales con cultura "en-US". Cruza con Mapa_Zonas.xlsx (hoja Zonas) usando la clave Text.Lower(Text.Trim(Distrito)) en ambos lados (LeftOuter); deja Distrito con la escritura de Mapa_Zonas y añade Zona; quita auxiliares. Latitud dataCategory Latitude y Longitud dataCategory Longitude, resumen Average.
4) dimUsuarios (M) desde Usuarios.csv (";"): IdUsuario y AnioNacimiento enteros; Genero: Text.Upper(Text.Trim) "M"/"MASCULINO" = "Masculino", "F"/"FEMENINO" = "Femenino", resto o vacío = "No especificado"; RangoEdad con Edad = 2025 - AnioNacimiento: <=24 "18-24", <=34 "25-34", <=44 "35-44", resto "45+", null "No especificado".
5) dimBicicletas (M) desde Bicicletas.xlsx (hoja Bicicletas, encabezados en la primera fila): sin RegistroInterno; IdBicicleta entero; FechaAdquisicion date.
6) dimFecha (tabla calculada DAX):
VAR T = CALENDAR(DATE(YEAR(MIN(fctViajes[FechaInicio])),1,1), DATE(YEAR(MAX(fctViajes[FechaInicio])),12,31))
RETURN SELECTCOLUMNS(T, "Fecha", [Date], "Anio", YEAR([Date]), "MesNum", MONTH([Date]), "Mes", MID("EneFebMarAbrMayJunJulAgoSepOctNovDic", (MONTH([Date])-1)*3+1, 3), "Trimestre", "T" & QUARTER([Date]), "AnioMes", FORMAT([Date], "yyyy-mm"))
Fecha tipo date, Mes ordenado por MesNum, Anio y MesNum sin resumir. Márcala como tabla de fechas (columna Fecha).
7) Actualiza las tablas y crea las relaciones (muchos a uno, una dirección): fctViajes[IdEstacionOrigen] → dimEstaciones[IdEstacion] ACTIVA; fctViajes[IdEstacionDestino] → dimEstaciones[IdEstacion] INACTIVA; fctViajes[IdBicicleta] → dimBicicletas[IdBicicleta]; fctViajes[IdUsuario] → dimUsuarios[IdUsuario]; fctViajes[FechaInicio] → dimFecha[Fecha]. Oculta las columnas Id* de fctViajes.
Notas para las herramientas: crea RutaDatos con named_expression_operations Create (expresión M "...ruta..." meta [IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true]); al crear tablas M con table_operations define explícitamente todas sus columnas (nombre y dataType) porque no se infieren; crea las 4 tablas M en una sola llamada y después dimFecha.
Si algo falla, corrígelo tú. Responde solo con: filas de cada tabla y relaciones (activa/inactiva).
```

2. Verificar las filas que reporta Claude: `fctViajes` 6.200, `dimEstaciones` 24, `dimUsuarios` 500, `dimBicicletas` 120 y `dimFecha` 1.096.

3. Si Power BI muestra la barra *"One or more relationships have been modified and need to be refreshed manually"*, pulsar **Refresh now**.

### A. _Validación de Arquitectura en Power BI_

4. Cambiar a la **Vista de modelo** (tercer icono de la barra lateral izquierda) y arrastrar las tablas para dejar `fctViajes` en el centro.

5. Verificar que `fctViajes` se conecta con cardinalidad de 1 a varios (`1:*`) a las cuatro dimensiones y que la relación `IdEstacionDestino → dimEstaciones` aparece como **línea punteada** (relación inactiva). Ver [Figura 33](#fig:33).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-02-modelo-estrella.png' width='700px'><br>
<caption><b>Figura 33<a name="fig:33"></a>.</b> Vista de modelo con el esquema en estrella. </caption>
</center>

6. Guardar con `Ctrl + S`.

> **Concepto clave:**
> El esquema en estrella separa los **hechos** (`fct`, cada viaje) de las **dimensiones** (`dim`: quién, qué, dónde y cuándo). Las relaciones `1:*` propagan el filtro desde la dimensión hacia los hechos. Entre dos tablas solo puede existir una relación activa; la de estación de destino queda inactiva y se activa dentro de una medida con `USERELATIONSHIP`. Marcar `dimFecha` como tabla de fechas habilita la inteligencia de tiempo (`TOTALYTD`, `DATEADD`, `DATESINPERIOD`), y el parámetro `RutaDatos` permite reubicar la carpeta de datos cambiando un solo valor.

## 3.4. Implementación y Organización de Medidas DAX en Tabla Dedicada

&nbsp; Siguiendo las directrices de diseño de Business Intelligence, se centralizan todas las medidas de cálculo en un contenedor independiente (`_Medidas`), clasificadas en carpetas de visualización (*Display Folders*). El prompt incluye el DAX completo para que Claude solo lo cree.

1. En el mismo chat, pegar el prompt:

**PROMPT 4 (Creación y Jerarquización de Medidas DAX)**
```text
Crea una tabla calculada "_Medidas" = {BLANK()} con su columna oculta, y en ella estas medidas EXACTAS en UNA sola llamada de measure_operations (displayFolder entre corchetes; formato después de "|"):
[Base]
Total Viajes = COUNTROWS(fctViajes) | #,0
Distancia Total (km) = SUM(fctViajes[DistanciaKm]) | #,0
Distancia Promedio (km) = AVERAGE(fctViajes[DistanciaKm]) | #,0.00
Duración Promedio (min) = AVERAGE(fctViajes[DuracionMin]) | #,0.0
Usuarios Activos = DISTINCTCOUNT(fctViajes[IdUsuario]) | #,0
Bicicletas en Uso = DISTINCTCOUNT(fctViajes[IdBicicleta]) | #,0
Viajes por Usuario = DIVIDE([Total Viajes], [Usuarios Activos]) | #,0.0
[Avanzadas]
% Viajes Eléctricos = DIVIDE(CALCULATE([Total Viajes], dimBicicletas[Tipo] = "Eléctrica"), [Total Viajes]) | 0.0%
Viajes por Destino = CALCULATE([Total Viajes], USERELATIONSHIP(fctViajes[IdEstacionDestino], dimEstaciones[IdEstacion])) | #,0
Viajes Año Anterior = VAR A = YEAR(MAX(fctViajes[FechaInicio])) RETURN CALCULATE([Total Viajes], dimFecha[Anio] = A - 1) | #,0
Var. Viajes vs AA = VAR A = YEAR(MAX(fctViajes[FechaInicio])) VAR Act = CALCULATE([Total Viajes], dimFecha[Anio] = A) VAR Ant = [Viajes Año Anterior] RETURN IF(NOT ISBLANK(Ant), Act - Ant) | +#,0;-#,0;0
Var. % Viajes vs AA = VAR A = YEAR(MAX(fctViajes[FechaInicio])) VAR Ant = [Viajes Año Anterior] RETURN DIVIDE(CALCULATE([Total Viajes], dimFecha[Anio] = A) - Ant, Ant) | +0.0%;-0.0%;0.0%
[Tendencias]
Viajes Acumulados Año = TOTALYTD([Total Viajes], dimFecha[Fecha]) | #,0
Viajes Media Móvil 3M = CALCULATE(AVERAGEX(VALUES(dimFecha[AnioMes]), [Total Viajes]), DATESINPERIOD(dimFecha[Fecha], MAX(dimFecha[Fecha]), -3, MONTH)) | #,0
Viajes Mes Anterior = CALCULATE([Total Viajes], DATEADD(dimFecha[Fecha], -1, MONTH)) | #,0
Var. % Viajes vs Mes Anterior = VAR U = MAX(fctViajes[FechaInicio]) VAR Ini = DATE(YEAR(U), MONTH(U), 1) VAR Act = CALCULATE([Total Viajes], DATESBETWEEN(dimFecha[Fecha], Ini, EOMONTH(U, 0))) VAR Ant = CALCULATE([Total Viajes], DATESBETWEEN(dimFecha[Fecha], EDATE(Ini, -1), Ini - 1)) RETURN DIVIDE(Act - Ant, Ant) | +0.0%;-0.0%;0.0%
Si "Eléctrica" no coincide con los valores de dimBicicletas[Tipo], ajústalo. Responde solo con el valor sin filtros de Total Viajes, % Viajes Eléctricos y Var. % Viajes vs AA.
```

2. Verificar los valores de control: **Total Viajes 6,200**, **% Viajes Eléctricos 34.1%** y **Var. % Viajes vs AA +9.5%** (2025 frente a 2024).

3. Volver a la **Vista de informe** (primer icono de la barra lateral izquierda) y, en el panel **Data**, expandir `_Medidas` y sus carpetas **Avanzadas**, **Base** y **Tendencias**. Ver [Figura 34](#fig:34).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-panel-medidas.png' width='700px'><br>
<caption><b>Figura 34<a name="fig:34"></a>.</b> Panel Data con la tabla _Medidas y sus carpetas. </caption>
</center>

### A. _Protocolo de Sincronización_

4. Guardar con `Ctrl + S` (no hace falta cerrar Power BI).

> **Concepto clave:**
> Una tabla dedicada separa la lógica de cálculo de los datos, y las carpetas agrupan las medidas por propósito. Las medidas de año anterior toman el **último año con datos del contexto** (`YEAR(MAX(...))`): sin filtros comparan 2025 contra 2024 en lugar de sumar todos los años. `CALCULATE` modifica el contexto de filtro, `USERELATIONSHIP` activa la relación inactiva solo dentro de la medida y `DIVIDE` evita errores por división entre cero.

## 3.5. Sistema de Diseño Institucional y Especificaciones de Maquetación

&nbsp; Se aplican los tokens de diseño institucional (`Tema_EcoBici.json`) y la grilla base para una resolución Full HD (1920 x 1080 px). El diseño se construye **una sola vez** en la página "Resumen" (panel lateral, 5 segmentadores, cabecera y tarjeta de KPIs) y luego se duplica para las otras dos páginas, garantizando consistencia cromática, espaciado y navegación transversal.

### A. _Aplicación del Tema Institucional_

1. En Power BI, abrir la pestaña **Design** (*Diseño*) y pulsar **Import theme** (*Importar tema*). Ver [Figura 35](#fig:35).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-importar-tema.png' width='700px'><br>
<caption><b>Figura 35<a name="fig:35"></a>.</b> Pestaña Design con la opción Import theme. </caption>
</center>

2. Navegar a la carpeta `Datos_Fuente`, seleccionar `Tema_EcoBici.json` y pulsar **Abrir**. Ver [Figura 36](#fig:36).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-seleccionar-tema.png' width='700px'><br>
<caption><b>Figura 36<a name="fig:36"></a>.</b> Selección del archivo Tema_EcoBici.json. </caption>
</center>

3. Confirmar el mensaje **File successfully added** con **Got it** y guardar con `Ctrl + S`. Ver [Figura 37](#fig:37).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-tema-aplicado.png' width='700px'><br>
<caption><b>Figura 37<a name="fig:37"></a>.</b> Confirmación de tema importado. </caption>
</center>

### B. _Maquetación de la Página Base_

4. En el mismo chat de Claude, pegar el prompt:

**PROMPT 5 (Layout Base: Panel Lateral, Segmentadores, Cabecera y KPIs)**
```text
Ahora el informe. Usa solo la herramienta de archivos sobre la carpeta EcoBici_Lima.Report\definition del proyecto (búscala con la herramienta de archivos; lee solo pages\pages.json y el page.json existente). Reglas PBIR para este y los próximos prompts: cada visual va en pages\<pagina>\visuals\<nombre>\visual.json con "$schema" visualContainer 2.x, "name" igual a la carpeta, "position" {x,y,z,width,height}. Columnas {"Column":{"Expression":{"SourceRef":{"Entity":"<tabla>"}},"Property":"<col>"}} y medidas {"Measure":{"Expression":{"SourceRef":{"Entity":"_Medidas"}},"Property":"<medida>"}}, queryRef "<tabla>.<campo>". Gráficos: título visible en negrita, sin subtítulo. Textboxes y shapes: sin fondo, borde, sombra ni título.
1) La página existente: displayName "Resumen", 1920 x 1080, fondo #EEF2F6.
2) Panel: shape "panelFondo" x0 y0 300x1080 relleno #1B4965; shape "panelFranja" x0 y0 300x6 relleno #F6AE2D.
3) Textboxes: "txtMarca" "EcoBici Lima" blanco 26px negrita (x24 y30 w252 h40); "txtLema" "Red de bicicletas públicas" #BEE3F8 13px (x24 y72 w252 h24); "txtFiltros" "FILTROS" #F6AE2D 12px negrita (x24 y130 w252 h24).
4) Slicers Dropdown con título, x24 w252 h80: "slAnio" y170 "Año" dimFecha[Anio]; "slMembresia" y266 "Tipo de membresía" dimUsuarios[TipoMembresia]; "slZona" y362 "Zona de origen" dimEstaciones[Zona]; "slTipoBici" y458 "Tipo de bicicleta" dimBicicletas[Tipo]; "slGenero" y554 "Género" dimUsuarios[Genero]. Cada uno con "syncGroup": {"groupName": "<columna>", "fieldChanges": true, "filterChanges": true} dentro de "visual".
5) Cabecera: "txtTitulo" "EcoBici Lima · Resumen Operacional" #1B4965 30px Segoe UI Semibold (x324 y24 w1572 h44); "txtSubtitulo" "Desempeño de la red de bicicletas públicas 2023-2025" #616E7C 15px (x324 y70 w1572 h26); shape "lineaAcento" x324 y102 96x5 relleno #F6AE2D.
6) cardVisual "kpis" x324 y131 w1572 h150 sin título: Total Viajes, Usuarios Activos, Distancia Total (km), Duración Promedio (min), % Viajes Eléctricos; valores #1B4965 30pt, displayUnits 1 (6,200 y no 6K), barra de acento #F6AE2D a la izquierda de cada valor.
Responde solo con "Listo".
```

### C. _Protocolo de Sincronización_

5. Cuando Claude responda "Listo", Power BI muestra la barra *"This project's files were changed externally"*. Pulsar **Apply external changes** (*Aplicar cambios externos*). Este mismo procedimiento se repite después de cada prompt de páginas. Ver [Figura 38](#fig:38).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-aplicar-cambios-externos.png' width='700px'><br>
<caption><b>Figura 38<a name="fig:38"></a>.</b> Barra Apply external changes en Power BI. </caption>
</center>

6. Si aparece el aviso **Overwrite your unsaved edits**, pulsar **Apply external changes**. Indica que había cambios sin guardar en Power BI; para evitarlo, guardar con `Ctrl + S` antes de pegar cada prompt y no guardar mientras Claude está escribiendo. Ver [Figura 39](#fig:39).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-sobrescribir-cambios.png' width='700px'><br>
<caption><b>Figura 39<a name="fig:39"></a>.</b> Aviso Overwrite your unsaved edits. </caption>
</center>

7. Revisar la página: panel lateral azul con los 5 segmentadores, cabecera y tarjeta de KPIs con los valores completos (6,200 y no 6K). Ver [Figura 40](#fig:40).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-04-diseno-base.png' width='700px'><br>
<caption><b>Figura 40<a name="fig:40"></a>.</b> Página base Resumen con panel lateral, cabecera y KPIs. </caption>
</center>

### D. _Sincronización y Duplicado de Páginas_

8. Seleccionar el segmentador **Año** y abrir **View → Sync slicers** (*Ver → Sincronizar segmentaciones*) para comprobar el grupo de sincronización. Ver [Figura 41](#fig:41).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-sincronizar-segmentadores.png' width='700px'><br>
<caption><b>Figura 41<a name="fig:41"></a>.</b> Panel Sync slicers con el segmentador Año sincronizado. </caption>
</center>

9. Clic derecho sobre la pestaña **Resumen** → **Duplicate** (*Duplicar*). Repetir una segunda vez. Ver [Figura 42](#fig:42).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-duplicar-pagina.png' width='700px'><br>
<caption><b>Figura 42<a name="fig:42"></a>.</b> Menú contextual de la pestaña con la opción Duplicate. </caption>
</center>

10. Doble clic sobre cada copia para renombrarlas como **Análisis de Uso** y **Tendencias**, y guardar con `Ctrl + S`. Ver [Figura 43](#fig:43).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-pestanas.png' width='700px'><br>
<caption><b>Figura 43<a name="fig:43"></a>.</b> Pestañas Resumen, Análisis de Uso y Tendencias. </caption>
</center>

> **Concepto clave:**
> En formato PBIR cada página es una carpeta y cada visual un archivo `visual.json`: Claude escribe esos archivos con el servidor `archivos` y Power BI los incorpora con **Apply external changes**, sin cerrar el programa. El tema JSON centraliza colores, tipografías, bordes y sombras de todos los visuales. El `syncGroup` vincula los segmentadores de distintas páginas: al duplicar la página, las copias conservan el mismo grupo y el filtro se comparte en todo el informe.

## 3.6. Construcción de la Página: Resumen Operacional

&nbsp; Despliegue de la primera vista analítica del informe, enfocada en los macroindicadores de operación y distribución espacial de la demanda. La cabecera y los KPIs ya existen; se agregan los cuatro gráficos.

1. Ir a la página **Resumen** y guardar con `Ctrl + S`.

2. En el mismo chat, pegar el prompt:

**PROMPT 6 (Gráficos de la Página "Resumen")**
```text
En la página "Resumen" crea (mismas reglas PBIR; no leas otros archivos salvo pages.json):
1) lineChart "lineasMes" x324 y305 w1032 h360 "Viajes por mes · 2024 vs 2025": Category dimFecha[Mes], Series dimFecha[Anio], Y Total Viajes; filtro del visual dimFecha[Anio] in (2024, 2025); líneas de 3 px con marcadores; color de 2024 #1B4965 y de 2025 #F6AE2D.
2) donutChart "donaMembresia" x1380 y305 w516 h360 "Viajes por tipo de membresía": Category dimUsuarios[TipoMembresia], Y Total Viajes; leyenda abajo; etiquetas en % del total; sin valor central ("centerValue" show false).
3) barChart "topEstaciones" x324 y689 w774 h367 "Top 10 estaciones de origen por viajes": Category dimEstaciones[NombreEstacion], Y Total Viajes; filtro TopN 10 por Total Viajes; barras #1B4965; etiquetas de datos; orden descendente.
4) azureMap "mapaEstaciones" x1122 y689 w774 h367 "Estaciones · tamaño según viajes de origen": Category dimEstaciones[NombreEstacion], Y dimEstaciones[Latitud] y X dimEstaciones[Longitud] con Aggregation Function 1 (promedio), Size Total Viajes; burbujas #F6AE2D.
Responde solo con "Listo".
```

3. Pulsar **Apply external changes** en Power BI.

4. Revisar la página terminada y guardar con `Ctrl + S`. Ver [Figura 44](#fig:44).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-05-resumen.png' width='700px'><br>
<caption><b>Figura 44<a name="fig:44"></a>.</b> Página Resumen Operacional terminada. </caption>
</center>

5. Si el gráfico de barras muestra todas las estaciones, aplicar el filtro manualmente: seleccionarlo → panel **Filters** → `NombreEstacion` → **Top N** → Mostrar **10** → arrastrar `Total Viajes` a *By value* → **Apply filter**.

> **Concepto clave:**
> El filtro de nivel de visual (2024 y 2025) solo afecta a ese gráfico. El filtro **Top N** se evalúa contra una medida, por lo que el ranking se recalcula con cada segmentador. En el mapa, latitud y longitud se agregan como **promedio**: si se sumaran, una estación con varias filas quedaría fuera de su ubicación real. Azure Maps requiere tener la sesión iniciada en Power BI Desktop.

## 3.7. Construcción de la Página: Análisis de Uso

&nbsp; Despliegue de la segunda página de analítica granular, orientada a estudiar el comportamiento por franjas horarias, segmentación demográfica y flujo de estaciones. La página ya existe como copia de "Resumen": el prompt cambia textos y KPIs, retira los gráficos copiados y crea los nuevos.

1. Ir a la página **Análisis de Uso** y guardar con `Ctrl + S`.

2. En el mismo chat, pegar el prompt:

**PROMPT 7 (Generación de la Página "Análisis de Uso")**
```text
En la página "Análisis de Uso" (copia de Resumen; búscala en pages.json, lista sus visuales y lee solo los que modifiques; no toques panel ni slicers):
1) Título: "EcoBici Lima · Análisis de Uso". Subtítulo: "Patrones de demanda por hora, mes, zona, usuarios y estaciones".
2) cardVisual: cambia las medidas por Viajes por Usuario, Distancia Promedio (km), Bicicletas en Uso, Viajes Año Anterior y Var. % Viajes vs AA (mismo formato).
3) Borra las carpetas de los visuales lineChart, donutChart, barChart y azureMap copiados.
4) Crea:
- columnChart "colHora" x324 y305 w774 h360 "Viajes por hora del día": Category fctViajes[HoraInicio] (orden ascendente), Y Total Viajes, columnas #1B4965.
- lineClusteredColumnComboChart "comboMes" x1122 y305 w774 h360 "Viajes y duración promedio por mes": Category dimFecha[Mes], Y Total Viajes (columnas), Y2 Duración Promedio (min) (línea).
- columnChart apilado "zonaTipo" x324 y689 w508 h367 "Viajes por zona de origen y tipo de bici": Category dimEstaciones[Zona], Series dimBicicletas[Tipo], Y Total Viajes.
- clusteredBarChart "edadGenero" x856 y689 w508 h367 "Viajes por rango de edad y género": Category dimUsuarios[RangoEdad], Series dimUsuarios[Genero], Y Total Viajes.
- tableEx "tablaEstaciones" x1388 y689 w508 h367 "Estaciones: salidas vs llegadas": NombreEstacion como "Estación", Zona como "Zona", Total Viajes como "Salidas", Viajes por Destino como "Llegadas"; encabezado fondo #1B4965 y texto blanco; orden descendente por Salidas.
Responde solo con "Listo".
```

3. Pulsar **Apply external changes** en Power BI.

4. Revisar la página y guardar con `Ctrl + S`. Ver [Figura 45](#fig:45).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-06-analisis-uso.png' width='700px'><br>
<caption><b>Figura 45<a name="fig:45"></a>.</b> Página Análisis de Uso terminada. </caption>
</center>

> **Concepto clave:**
> Al partir de una copia, el panel lateral y los segmentadores sincronizados se conservan sin reescribirse. El gráfico combinado usa **dos ejes**: columnas para el volumen de viajes y línea para la duración en minutos, magnitudes con escalas distintas. La tabla contrasta la relación activa (salidas por estación de origen) con la inactiva activada por `USERELATIONSHIP` (llegadas por estación de destino).

## 3.8. Construcción de la Página: Tendencias

&nbsp; Generación de la tercera vista, especializada en series de tiempo, tasas de variación y evolución de la mezcla de flotas. Se sigue el mismo procedimiento que en la sección 3.7.

1. Ir a la página **Tendencias** y guardar con `Ctrl + S`.

2. En el mismo chat, pegar el prompt:

**PROMPT 8 (Generación de la Página "Tendencias")**
```text
En la página "Tendencias" (copia de Resumen; mismo procedimiento que en Análisis de Uso):
1) Título: "EcoBici Lima · Tendencias". Subtítulo: "Evolución mensual 2023-2025: volumen, ritmo de crecimiento y composición".
2) cardVisual: Viajes Acumulados Año, Viajes Media Móvil 3M, Var. % Viajes vs Mes Anterior y Var. % Viajes vs AA.
3) Borra los visuales lineChart, donutChart, barChart y azureMap copiados.
4) Crea:
- lineChart "lineasMediaMovil" x324 y305 w1572 h360 "Viajes mensuales y media móvil de 3 meses": Category dimFecha[AnioMes] (orden ascendente), Y Total Viajes y Viajes Media Móvil 3M.
- lineChart "acumuladoAnio" x324 y689 w508 h367 "Viajes acumulados en el año": Category dimFecha[Mes], Series dimFecha[Anio], Y Viajes Acumulados Año.
- hundredPercentStackedColumnChart "mezclaFlota" x856 y689 w508 h367 "Mezcla anual: eléctricas vs mecánicas": Category dimFecha[Anio], Series dimBicicletas[Tipo], Y Total Viajes, etiquetas de datos.
- clusteredColumnChart "viajesTrimestre" x1388 y689 w508 h367 "Viajes por trimestre y año": Category dimFecha[Trimestre], Series dimFecha[Anio], Y Total Viajes.
Responde solo con "Listo".
```

3. Pulsar **Apply external changes** en Power BI.

4. Revisar la página y guardar con `Ctrl + S`. Ver [Figura 46](#fig:46).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-07-tendencias.png' width='700px'><br>
<caption><b>Figura 46<a name="fig:46"></a>.</b> Página Tendencias terminada. </caption>
</center>

> **Concepto clave:**
> La **media móvil de 3 meses** suaviza la variación mensual para mostrar la tendencia de fondo. El **acumulado del año** (`TOTALYTD`) se reinicia en enero, por eso cada año es una curva creciente. Las columnas **100 % apiladas** muestran proporciones (eléctricas frente a mecánicas) y no volumen, lo que permite comparar años con distinta cantidad de viajes.

## 3.9. Validación Técnica Integral y Control de Calidad

&nbsp; Concluida la construcción programática, se ejecutan las pruebas de integración del artefacto analítico:

### A. _Persistencia de contexto de filtrado_

1. En cualquier página, abrir el segmentador **Año** y seleccionar **2024**. Ver [Figura 47](#fig:47).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/paso-validar-filtro.png' width='700px'><br>
<caption><b>Figura 47<a name="fig:47"></a>.</b> Segmentador Año desplegado. </caption>
</center>

2. Cambiar a otra página: el segmentador debe mostrar **2024** y los KPIs y gráficos deben reflejar solo ese año (por ejemplo, Total Viajes 2,100). Ver [Figura 48](#fig:48).

<center>
<img src='https://raw.githubusercontent.com/eleventhcurve91/Img-repo/main/claude-free/free-08-validacion-filtros.png' width='700px'><br>
<caption><b>Figura 48<a name="fig:48"></a>.</b> Filtro 2024 sincronizado en la página Resumen. </caption>
</center>

3. Pasar el cursor sobre el segmentador y pulsar el icono de borrador (**Clear selections**) para quitar el filtro.

### B. _Evaluación de relaciones no activas_

4. En **Análisis de Uso**, comprobar que la columna **Llegadas** de la tabla de estaciones da valores distintos de **Salidas**.

### C. _Persistencia final del repositorio PBIP_

5. Guardar con `Ctrl + S`. La carpeta `EcoBici_Lima.Report` contiene las tres páginas en formato PBIR, preparadas para control de versiones o despliegue en Fabric / Power BI Service.

> **Concepto clave:**
> Validar el filtrado cruzado confirma que el `syncGroup` funciona en las tres páginas y que todas las medidas responden al contexto de filtro. La diferencia entre salidas y llegadas confirma que la relación inactiva se activa correctamente con `USERELATIONSHIP`.

# 4. Resolución de Incidencias Técnicas (Troubleshooting)

&nbsp; Guía técnica de diagnóstico y resolución de eventos comunes en estaciones de trabajo:

| Escenario o Mensaje de Error | Causa Raíz Probable | Procedimiento de Mitigación |
| :--- | :--- | :--- |
| **`powerbi-modeling-mcp` en estado "Error" o "Stopped"** | Alternativa A: Node.js no está en el PATH o `npx` no logra descargar el paquete. Alternativa B: la ruta al ejecutable no existe (extensión no instalada o carpeta con otra versión). En ambas: error de sintaxis en `claude_desktop_config.json`. | Comprobar `node -v` y `npx -v`. En la alternativa B, ubicar el `.exe` con el comando de PowerShell indicado y duplicar cada barra invertida. Validar el archivo con un validador JSON estándar. |
| **`archivos` en estado "Error" o "Stopped"** | Node.js no está expuesto en el PATH del sistema o la carpeta indicada en el último argumento no existe. | Comprobar `node -v` y `npx -v` en terminal, y verificar que la ruta de la carpeta de trabajo exista y tenga las barras duplicadas. |
| **Tiempo de espera agotado en comandos `npx`** | Restricciones de red perimetral, proxies con inspección TLS o políticas de endpoint. | Usar la alternativa B para el servidor de Power BI, que no descarga el paquete en cada arranque. |
| **Alternativa B: el servidor de modelado dejó de funcionar tras actualizar VS Code** | El nombre de la carpeta de la extensión incluye la versión (`...powerbi-modeling-mcp-1.0.0-win32-x64`) y cambia al actualizarse. | Volver a ubicar el ejecutable y actualizar la ruta en el campo `command` del servidor `powerbi-modeling-mcp`. |
| **Fallo de enlace MCP con el informe en ejecución** | Proceso de Power BI Desktop cerrado o acceso de herramientas externas inactivo. | Abrir Power BI con el proyecto `.pbip` correspondiente. Verificar en **Archivo > Opciones y configuración > Opciones > Características en versión preliminar** que el acceso de herramientas externas mediante API locales seguras esté activado. |
| **Los componentes visuales no renderizan tras el prompt** | No se confirmó el diálogo de recarga generado por el Desktop Bridge en Power BI Desktop. | Pulsar el botón **Aceptar** o **Recargar** en el diálogo modal emitido por Power BI Desktop y consolidar con `Ctrl + S`. |
| **Diálogo modal de error de renderizado en gráficos** | Desfase temporal en el refresco de componentes por el bridge de Power BI. | Hacer clic en **Cerrar** en el cuadro de diálogo de Power BI. No afecta el modelo semántico ni la estructura del reporte. |
| **Mapa de Azure Maps bloqueado o en blanco** | Ausencia de sesión activa de usuario en Power BI Desktop o mapas no habilitados. | Iniciar sesión en Power BI Desktop con una cuenta Microsoft/empresarial en la esquina superior derecha y verificar la habilitación de mapas en opciones preliminares. |
| **Error en resolución de ruta de Datos_Fuente** | Discrepancia entre la ruta asignada al parámetro `RutaDatos` y la ubicación real del directorio en disco. | Validar la ruta absoluta del directorio `PowerBI` y verificar la asignación correcta del parámetro `RutaDatos` en el modelo. |